# Flatten orders
Project nested book deltas into the canonical `Order` table with Arrow kernels.

In [ ]:
source = "market.books"
target = "market.orders"
start = None
end = None
catalog = "rekep"
catalog_properties = {}
branch = "root"
merge_by = True
commit_row_size = 250_000

In [ ]:
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThan
from rekep.iceberg import IcebergDataset
from rekep.market import Book, Order
from rekep.times import unix_of


def _filter(column="unix"):
    lower, upper = unix_of(start), unix_of(end, upper=True)
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


books = IcebergDataset(
    name=source, catalog=catalog, properties=dict(catalog_properties), branch=branch
)
orders = IcebergDataset(
    name=target,
    catalog=catalog,
    properties=dict(catalog_properties),
    branch=branch,
    field=Order.into_field(),
    commit_row_size=commit_row_size,
    sort_by=("unix", "hash"),
)
counts = {"read": 0}


def _batches():
    reader = books.read_arrow_reader(
        Book.into_field(), row_filter=_filter(), order_by=("unix", "hash")
    )
    for batch in reader:
        flattened = Order.from_books_arrow_batch(batch)
        counts["read"] += flattened.num_rows
        if flattened.num_rows:
            yield flattened


written = orders.append_arrow_reader(
    _batches(), Order.into_field(), merge_by=merge_by, commit_row_size=commit_row_size
)
result = {"read": counts["read"], "written": written, "target": target}
try:
    import scrapbook as sb
except ImportError:
    pass
else:
    sb.glue("result", result, encoder="json")
result